# Proyecto Parte 1 — Clasificación de Textos por Década
## Competencia Kaggle · Aprendizaje de Máquina 2026-10

En este notebook abordamos la **Parte 1** de la competencia del curso: clasificar párrafos de texto histórico en español según la **década en que fueron escritos**, empleando exclusivamente técnicas clásicas de aprendizaje automático (sin redes neuronales ni arquitecturas *transformer*).

La variable objetivo es la **década**, representada con los tres primeros dígitos del año (e.g., el año 1572 corresponde a la década `157`). Las clases van de `150` a `188`, para un total de **39 clases**.

## 1. Importación de librerías

In [ ]:
import pickle
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

## 2. Carga de los datos

In [ ]:
train = pd.read_csv('./data/train.csv')
eval_df = pd.read_csv('./data/eval.csv')

print(f'Entrenamiento : {train.shape[0]:,} filas, {train.shape[1]} columnas')
print(f'Evaluación    : {eval_df.shape[0]:,} filas, {eval_df.shape[1]} columnas')
train.head()

In [ ]:
print(f'Valores nulos en train : {train.isnull().sum().sum()}')
print(f'Valores nulos en eval  : {eval_df.isnull().sum().sum()}')

## 3. Exploración de los datos

In [ ]:
print(f'Clases únicas    : {train["decade"].nunique()}')
print(f'Rango de décadas : {train["decade"].min()} – {train["decade"].max()}')

conteo = train['decade'].value_counts().sort_index()
print(f'\nMedia por clase  : {conteo.mean():.1f}')
print(f'Mínimo           : {conteo.min()} (década {conteo.idxmin()})')
print(f'Máximo           : {conteo.max()} (década {conteo.idxmax()})')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(conteo.index.astype(str), conteo.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Década')
ax.set_ylabel('Cantidad de textos')
ax.set_title('Distribución de textos por década')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
train['text_len'] = train['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train['text_len'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Longitud (caracteres)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitud de textos')

axes[1].boxplot(train['text_len'], vert=False)
axes[1].set_xlabel('Longitud (caracteres)')
axes[1].set_title('Boxplot de longitud de textos')
plt.tight_layout()
plt.show()

print(f'Media   : {train["text_len"].mean():.1f} caracteres')
print(f'Mediana : {train["text_len"].median():.1f} caracteres')
train = train.drop(columns=['text_len'])

In [ ]:
for dec in [150, 162, 175, 188]:
    sample = train[train['decade'] == dec]['text'].iloc[0]
    print(f'--- Década {dec} (años {dec}0–{dec}9) ---')
    print(sample[:250].strip())
    print()

## 4. Preprocesamiento de texto

Los textos contienen saltos de línea múltiples y espacios redundantes producto del OCR. Se aplica una limpieza mínima: conversión a minúsculas y colapso de espacios.

Se decide **no eliminar** puntuación ni palabras de parada porque en textos históricos estas formas pueden ser indicadores de época (e.g., "vuesa merced", uso de la `f` larga como `s`). Preservar ese vocabulario entrega información discriminativa al modelo.

In [ ]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = ' '.join(texto.split())
    return texto

train['text_clean'] = train['text'].apply(limpiar_texto)
eval_df['text_clean'] = eval_df['text'].apply(limpiar_texto)

print('Antes  :', repr(train['text'].iloc[1][:120]))
print('Después:', repr(train['text_clean'].iloc[1][:120]))

## 5. Partición del conjunto de datos

Se separa un 15 % del conjunto de entrenamiento como validación, usando `stratify=y` para mantener la proporción de clases en ambas particiones.

In [ ]:
X = train['text_clean'].values
y = train['decade'].values
X_eval = eval_df['text_clean'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

print(f'Entrenamiento : {X_train.shape[0]:,} textos')
print(f'Test          : {X_test.shape[0]:,} textos')
print(f'Evaluación    : {X_eval.shape[0]:,} textos')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

---

## Actividad 1: TF-IDF + LinearSVC con búsqueda de hiperparámetros

Se construye un pipeline con `TfidfVectorizer` y `LinearSVC`. El vectorizador convierte cada texto en un vector de pesos TF-IDF; `LinearSVC` entrena un clasificador de margen máximo para las 39 clases bajo el esquema *one-vs-rest*.

Se usa `sublinear_tf=True` (transformación `tf → 1 + log(tf)`) para reducir el peso desproporcionado de términos muy repetidos en un mismo documento, práctica estándar en clasificación de texto. El espacio de búsqueda explora:
- `ngram_range`: unigramas vs. unigramas+bigramas.
- `max_features`: tamaño del vocabulario retenido.
- `C`: parámetro de regularización de `LinearSVC` — valores bajos imponen mayor margen (más regularización), valores altos toleran más errores de clasificación.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, min_df=2)),
    ('clf',   LinearSVC(max_iter=3000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2)],
    'tfidf__max_features': [100_000, 150_000],
    'clf__C': [0.5, 1.0, 5.0],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

In [ ]:
%%time
grid.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros:')
for k, v in grid.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor F1 macro en CV (5-fold): {grid.best_score_:.4f}')

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
cv_results[['param_tfidf__ngram_range', 'param_tfidf__max_features', 'param_clf__C',
            'mean_train_score', 'mean_test_score', 'std_test_score']] \
    .sort_values('mean_test_score', ascending=False) \
    .reset_index(drop=True)

In [ ]:
best_model = grid.best_estimator_

y_train_pred = best_model.predict(X_train)
print('------ Mejor modelo - Entrenamiento ------')
print(f'Accuracy : {accuracy_score(y_train, y_train_pred):.4f}')
print(f'F1 macro : {f1_score(y_train, y_train_pred, average="macro"):.4f}')

In [ ]:
y_test_pred = best_model.predict(X_test)
print('------ Mejor modelo - Test ------')
print(f'Accuracy : {accuracy_score(y_test, y_test_pred):.4f}')
print(f'F1 macro : {f1_score(y_test, y_test_pred, average="macro"):.4f}')
print()
print(classification_report(y_test, y_test_pred, zero_division=0))

---

## Actividad 2: Curvas de validación

Se generan curvas de validación para visualizar cómo evoluciona el F1 macro (en entrenamiento y en validación cruzada) al variar el parámetro `C` de `LinearSVC`, fijando el resto de hiperparámetros en la configuración óptima encontrada en la Actividad 1. La banda de color representa la desviación estándar entre los 5 folds.

In [ ]:
best_ngram    = grid.best_params_['tfidf__ngram_range']
best_features = grid.best_params_['tfidf__max_features']

pipeline_vc = Pipeline([
    ('tfidf', TfidfVectorizer(
        sublinear_tf=True,
        min_df=2,
        ngram_range=best_ngram,
        max_features=best_features
    )),
    ('clf', LinearSVC(max_iter=3000, random_state=42))
])

param_range = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]

train_scores, val_scores = validation_curve(
    estimator=pipeline_vc,
    X=X_train,
    y=y_train,
    param_name='clf__C',
    param_range=param_range,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1
)

In [ ]:
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(param_range, train_mean, marker='o', label='Entrenamiento', color='steelblue')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.2, color='steelblue')
plt.plot(param_range, val_mean, marker='o', label='Validación cruzada', color='coral')
plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.2, color='coral')
plt.xscale('log')
plt.xlabel('C (escala logarítmica)')
plt.ylabel('F1 macro')
plt.title('Curvas de validación — parámetro C de LinearSVC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación sesgo-varianza:**

- **C pequeño:** alta regularización. Si ambas curvas tienen F1 bajo y similar, hay **alto sesgo** (subajuste): el modelo es demasiado restrictivo para capturar la variabilidad léxica entre décadas.
- **C óptimo:** las curvas convergen con F1 alto. La brecha pequeña entre entrenamiento y validación indica que el modelo generaliza bien.
- **C grande:** el F1 en entrenamiento sigue subiendo pero el de validación se estanca o baja, lo que indica **sobreajuste** — el modelo memoriza el conjunto de entrenamiento sin generalizar.

---

## Actividad 3: Análisis de resultados y selección del modelo

Se consolidan los mejores resultados de la búsqueda y se analiza el desempeño por clase para identificar qué décadas son más difíciles de clasificar.

In [ ]:
idx = grid.best_index_
resultados = pd.DataFrame({
    'Métrica'           : ['F1 macro CV (media)', 'F1 macro CV (std)', 'F1 macro Test', 'Accuracy Test'],
    'Valor'             : [
        round(grid.cv_results_['mean_test_score'][idx], 4),
        round(grid.cv_results_['std_test_score'][idx],  4),
        round(f1_score(y_test, y_test_pred, average='macro'), 4),
        round(accuracy_score(y_test, y_test_pred), 4),
    ]
})
resultados

In [ ]:
clases    = sorted(set(y_test))
f1_xclase = f1_score(y_test, y_test_pred, average=None, labels=clases)

f1_df = pd.DataFrame({'Década': clases, 'F1': f1_xclase}).sort_values('F1')

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(f1_df['Década'].astype(str), f1_df['F1'], color='steelblue', edgecolor='white')
ax.axhline(f1_df['F1'].mean(), color='red', linestyle='--', label=f'Media: {f1_df["F1"].mean():.3f}')
ax.set_xlabel('Década')
ax.set_ylabel('F1')
ax.set_title('F1 por clase en el conjunto de test')
ax.tick_params(axis='x', rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

print('Décadas con menor F1:')
print(f1_df.head(5).to_string(index=False))

---

## Generación del archivo de envío y guardado del modelo

In [ ]:
preds_eval = best_model.predict(X_eval)
submission = pd.DataFrame({'id': eval_df['id'], 'answer': preds_eval})
submission.to_csv('./data/submission.csv', index=False)
print('submission.csv guardado.')
submission.head()

In [ ]:
joblib.dump(best_model, './data/modelo_final.joblib')
print('Modelo guardado en: ./data/modelo_final.joblib')

# Verificación
modelo_cargado = joblib.load('./data/modelo_final.joblib')
check = modelo_cargado.predict(X_test[:5])
print(f'Predicciones de prueba : {check}')
print(f'Etiquetas reales       : {y_test[:5]}')

---

## Análisis de resultados

**¿Cuál configuración obtuvo el mejor desempeño y por qué?**

> *Completar con los valores de la tabla de la Actividad 3. Identificar el ngram_range, max_features y C óptimos, y argumentar por qué esa combinación funciona mejor para textos históricos.*

---

**¿Qué décadas son más difíciles de clasificar y por qué?**

> *Observar las décadas con menor F1 en la gráfica de la Actividad 3. Décadas contiguas (e.g., 160 y 161) tienden a confundirse porque el vocabulario cambia gradualmente. Décadas muy separadas en el tiempo suelen ser más fáciles de distinguir.*

---

**¿Se observa sobreajuste?**

> *Comparar F1 macro en entrenamiento vs. test. Si la brecha es pequeña, el modelo generaliza correctamente. Revisar también las curvas de validación de la Actividad 2.*

---

## Uso de herramientas de IA generativa

**Herramienta utilizada:** Claude (Anthropic).

**Tipo de uso:** Generación del esqueleto inicial del notebook y apoyo conceptual sobre la elección de `LinearSVC` y `TfidfVectorizer` para clasificación de texto histórico.

**Prompts utilizados:**

- *'Genera el esqueleto de un notebook de clasificación de texto histórico en español usando TF-IDF y LinearSVC con GridSearchCV en scikit-learn, siguiendo el estilo de notebooks educativos del curso de Aprendizaje de Máquina.'*